# Migrate an IV notebook to the Rust engine

For existing IV / PyVISA Notebook users. Install the KohdaLab IV desktop or portable application and the thin client into this kernel first. This tutorial only uses simulation, with no hardware output. Keep your existing analysis and plotting cells; replace device control and acquisition cells.

1. Inspect automatic application discovery.
2. Replace legacy device objects with one Rust-owned session.
3. Save a sweep and load its CSV for existing analysis.
4. Check cleanup and plan hardware migration.


In [ ]:
from pathlib import Path
import csv
import tempfile
from kohdalab_iv_native import NativeIV, resolve_executable

# Windows: installed app. Mac: ~/Applications or /Applications/KohdaLab IV.app.
print("Selected Rust engine:", resolve_executable())
# If you keep a portable app elsewhere, pass its folder / .app to NativeIV:
# NativeIV(executable="/absolute/path/to/KohdaLab IV.app")


## Replace device-control cells

Remove old Python driver imports, ResourceManager creation and raw write/query loops. These are not API-compatible with the new client. Rust now owns the source, meter, integration, limits and output cleanup. Keep each acquisition in one context manager so a Notebook interrupt requests safe cleanup.

Source level uses SI units: 1 mV is `1e-3` V. `dc_iv` means source voltage / measure current; `dc_vi` means source current / measure voltage. In the run configuration, `stop` means the old End field. Inspect defaults rather than copying old configuration files blindly.


In [ ]:
output = Path(tempfile.mkdtemp(prefix="iv-migration-")) / "simulation.csv"
with NativeIV() as iv:
    iv.connect(simulation=True)
    iv.set_mode("dc_iv")
    defaults = iv.default_config
    print("Configuration fields:", sorted(defaults))
    sample = iv.live()
    print("Live sample:", sample)
    result = iv.run(output, {
        "mode": "dc_iv", "pattern": "linear",
        "start": -1e-3, "stop": 1e-3, "step": 1e-3,
        "compliance": 1e-3, "settle_s": 0.0, "start_settle_s": 0.0,
    })
    assert result["phase"] == "Completed"
    assert result["output"] == "Off"
assert iv.snapshot["output"] == "Off"
assert not iv.snapshot["connected"]
print("Run:", result["points"], "points; output", result["output"])


## Keep analysis cells

`run()` returns status, not a data table. Load the CSV for point-by-point analysis. Column names are available in `fieldnames`; choose them explicitly when adapting old plots. CSV values use SI units and scientific notation. Do not round them to match the GUI's display precision.


In [ ]:
with output.open(newline="") as stream:
    reader = csv.DictReader(stream)
    columns = reader.fieldnames
    rows = list(reader)
assert len(rows) == result["points"] == 3
print("Columns:", columns)
print("First row:", rows[0])
# Existing pandas analysis can use: frame = pandas.read_csv(output)
# Metadata and settings are saved alongside it in output.with_suffix(".run.json").


## Hardware migration checklist

Before using hardware, disconnect the GUI from the same devices, confirm resources, models, wiring and permitted limits. Change the connect call to `iv.connect(source_resource, meter_resource, source_model=..., meter_model=...)` only after those checks. Do not run old and new hardware sessions concurrently. `live()` reads without enabling source output; `run()` enables output for the sweep. Context exit waits for output cleanup; if cleanup reports an error, confirm physical output status before continuing.

## Exercise

Repeat the simulated sweep with `step=5e-4` and a new filename. Expect five rows. Keep simulation enabled.


In [ ]:
# with NativeIV() as iv:
#     iv.connect(simulation=True)
#     result2 = iv.run(output.with_name("second.csv"), {
#         "pattern": "linear", "start": -1e-3, "stop": 1e-3, "step": 5e-4,
#         "settle_s": 0.0, "start_settle_s": 0.0,
#     })
#     assert result2["points"] == 5 and result2["output"] == "Off"
